In [ ]:
plant_disease_efficientnet_b0.pth
best (1).pt

In [ ]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 460.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 8.9 MB/s eta 0:00:00


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving best (1).pt to best (1).pt


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving plant_disease_efficientnet_b0.pth to plant_disease_efficientnet_b0.pth


In [ ]:
import torch
import torchvision.models as models
from ultralytics import YOLO

# 🦠 Disease model
disease_model = models.efficientnet_b0(weights=None)

disease_model.classifier[1] = torch.nn.Linear(
    disease_model.classifier[1].in_features,
    38
)

disease_model.load_state_dict(
    torch.load(
        "/content/plant_disease_efficientnet_b0.pth",
        map_location="cpu"
    )
)

disease_model.eval()

# 🐛 Pest model
pest_model = YOLO("/content/best (1).pt")

print("✅ Disease model restored")
print("✅ Pest model restored")
print("🎯 Both models are ready!")

✅ Disease model restored
✅ Pest model restored
🎯 Both models are ready!


In [ ]:
import torch
import requests
import numpy as np
import folium
import gradio as gr
from torchvision import transforms
from sklearn.cluster import DBSCAN

print("✅ Required libraries ready!")

✅ Required libraries ready!


In [ ]:
disease_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

print("✅ Disease image preprocessing restored!")

✅ Disease image preprocessing restored!


In [ ]:
classes = [
    'Apple___Apple_scab',
    'Apple___Black_rot',
    'Apple___Cedar_apple_rust',
    'Apple___healthy',
    'Blueberry___healthy',
    'Cherry_(including_sour)___Powdery_mildew',
    'Cherry_(including_sour)___healthy',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn_(maize)___Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight',
    'Corn_(maize)___healthy',
    'Grape___Black_rot',
    'Grape___Esca_(Black_Measles)',
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
    'Grape___healthy',
    'Orange___Haunglongbing_(Citrus_greening)',
    'Peach___Bacterial_spot',
    'Peach___healthy',
    'Pepper,_bell___Bacterial_spot',
    'Pepper,_bell___healthy',
    'Potato___Early_blight',
    'Potato___Late_blight',
    'Potato___healthy',
    'Raspberry___healthy',
    'Soybean___healthy',
    'Squash___Powdery_mildew',
    'Strawberry___Leaf_scorch',
    'Strawberry___healthy',
    'Tomato___Bacterial_spot',
    'Tomato___Early_blight',
    'Tomato___Late_blight',
    'Tomato___Leaf_Mold',
    'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite',
    'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy'
]

print("✅ Classes restored:", len(classes))

✅ Classes restored: 38


In [ ]:
CONFIDENCE_THRESHOLD = 0.70

print("✅ Confidence threshold restored:", CONFIDENCE_THRESHOLD)

✅ Confidence threshold restored: 0.7


In [ ]:
def get_weather_risk(temperature, humidity, rainfall):

    if humidity >= 80 and rainfall >= 10:
        return "🔴 HIGH RISK"

    elif humidity >= 65 or rainfall >= 5:
        return "🟡 MEDIUM RISK"

    else:
        return "🟢 LOW RISK"


print("✅ Weather risk function restored!")

✅ Weather risk function restored!


In [ ]:
def calculate_overall_risk(
    disease_confidence,
    pest_confidence,
    weather_risk
):
    score = 0

    if disease_confidence < 0.70:
        score += 40
    elif disease_confidence < 0.85:
        score += 25
    else:
        score += 10

    if pest_confidence > 0:
        score += 30

    if "HIGH" in weather_risk:
        score += 30
    elif "MEDIUM" in weather_risk:
        score += 20
    else:
        score += 5

    if score >= 70:
        risk = "🔴 HIGH CROP RISK"
    elif score >= 40:
        risk = "🟡 MEDIUM CROP RISK"
    else:
        risk = "🟢 LOW CROP RISK"

    return score, risk


print("✅ Overall risk function restored!")

✅ Overall risk function restored!


In [ ]:
def generate_farmer_alert(risk_level):

    if "HIGH" in risk_level:
        return """
🚨 FARMER ALERT

⚠️ High crop risk detected!

🔍 Increase crop monitoring immediately.
🌱 Inspect affected plants.
🔬 Expert validation is recommended.
📞 Contact the nearest agricultural expert if symptoms spread.
"""

    elif "MEDIUM" in risk_level:
        return """
⚠️ FARMER ALERT

🟡 Medium crop risk detected.

🌱 Monitor the crop regularly.
🔍 Check for disease and pest symptoms.
🌦️ Continue monitoring weather conditions.
"""

    else:
        return """
✅ FARMER STATUS

🟢 Low crop risk detected.

🌱 Continue regular crop monitoring
and preventive crop management.
"""

print("✅ Farmer alert restored!")

✅ Farmer alert restored!


In [ ]:
def generate_advisory(disease, pest, weather_risk):
    advice = []

    if "healthy" not in disease.lower():
        advice.append(
            f"🦠 Disease: {disease}\n"
            "Remove affected plant parts and monitor the crop regularly."
        )
    else:
        advice.append(
            "🦠 Disease: No major disease indicated.\n"
            "Continue regular crop monitoring."
        )

    if pest != "No pest detected":
        advice.append(
            f"🐛 Pest: {pest}\n"
            "Inspect affected plants and consider appropriate integrated pest management."
        )
    else:
        advice.append(
            "🐛 Pest: No pest detected.\n"
            "Continue regular monitoring."
        )

    if "HIGH" in weather_risk:
        advice.append(
            "🌦️ Weather: HIGH RISK\n"
            "Increase field monitoring and consider expert validation."
        )
    elif "MEDIUM" in weather_risk:
        advice.append(
            "🌦️ Weather: MEDIUM RISK\n"
            "Monitor the crop closely during the coming period."
        )
    else:
        advice.append(
            "🌦️ Weather: LOW RISK\n"
            "Continue preventive crop management."
        )

    return "\n\n".join(advice)


print("✅ generate_advisory restored!")

✅ generate_advisory restored!


In [ ]:
language_options = [
    "English",
    "தமிழ்",
    "हिन्दी",
    "తెలుగు",
    "ಕನ್ನಡ",
    "മലയാളം",
    "বাংলা",
    "मराठी",
    "ગુજરાતી",
    "ਪੰਜਾਬੀ",
    "ଓଡ଼ିଆ"
]

print("✅ Language options restored!")

✅ Language options restored!


In [ ]:
def complete_analysis(img, language, latitude, longitude):

    if img is None:
        return (
            "⚠️ Please upload a crop image.",
            "",
            "",
            "",
            "",
            "",
            "⚠️ Please upload an image."
        )

    # 🦠 Disease detection
    image_tensor = disease_transform(img).unsqueeze(0)

    with torch.no_grad():
        output = disease_model(image_tensor)
        probs = torch.softmax(output, dim=1)
        confidence, predicted = torch.max(probs, 1)

    disease = classes[predicted.item()]
    disease_conf = confidence.item()

    # 🔬 Expert validation
    if disease_conf >= CONFIDENCE_THRESHOLD:
        validation = "✅ Prediction confidence acceptable."
    else:
        validation = (
            "⚠️ Uncertain prediction\n"
            "🔬 Expert validation recommended."
        )

    # 🐛 Pest detection
    pest_results = pest_model.predict(
        source=img,
        conf=0.25,
        verbose=False
    )

    pest_name = "No pest detected"
    pest_conf = 0

    if len(pest_results) > 0 and len(pest_results[0].boxes) > 0:
        box = pest_results[0].boxes[0]
        pest_conf = float(box.conf[0])
        pest_id = int(box.cls[0])
        pest_name = pest_results[0].names[pest_id]

    # 🌦️ Weather
    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={latitude}&longitude={longitude}"
        f"&current=temperature_2m,relative_humidity_2m,rain"
    )

    try:
        weather = requests.get(url, timeout=10).json()

        temperature = weather["current"]["temperature_2m"]
        humidity = weather["current"]["relative_humidity_2m"]
        rainfall = weather["current"]["rain"]

        weather_risk = get_weather_risk(
            temperature,
            humidity,
            rainfall
        )

    except Exception:
        temperature = "Unavailable"
        humidity = "Unavailable"
        rainfall = "Unavailable"
        weather_risk = "⚠️ Weather unavailable"

    # 📊 Overall risk
    risk_score, risk_level = calculate_overall_risk(
        disease_conf,
        pest_conf,
        weather_risk
    )

    # 🚨 Farmer alert
    farmer_alert = generate_farmer_alert(risk_level)

    # 💊 Advisory
    advisory = get_multilingual_advisory(
        disease,
        pest_name,
        weather_risk,
        language
    )

    # 🗺️ Farmer location map
    farmer_map = folium.Map(
        location=[latitude, longitude],
        zoom_start=13
    )

    folium.Marker(
        [latitude, longitude],
        popup="📍 Farmer Field Location",
        tooltip="Farmer Location"
    ).add_to(farmer_map)

    analysis = f"""
🌱 CROP HEALTH ANALYSIS

🦠 Disease:
{disease}

📊 Disease Confidence:
{disease_conf * 100:.2f}%

🐛 Pest:
{pest_name}

📊 Pest Confidence:
{pest_conf * 100:.2f}%

🌦️ LIVE WEATHER

🌡️ Temperature: {temperature} °C
💧 Humidity: {humidity} %
🌧️ Rainfall: {rainfall} mm

⚠️ Weather Risk:
{weather_risk}

📊 OVERALL CROP RISK

Risk Score: {risk_score}/100
Risk Level: {risk_level}

📍 FARMER LOCATION

Latitude: {latitude}
Longitude: {longitude}
"""

    status = "✅ Analysis completed successfully."

    return (
        analysis,
        advisory,
        validation,
        farmer_alert,
        f"{risk_level}\nRisk Score: {risk_score}/100",
        farmer_map._repr_html_(),
        status
    )


print("✅ Complete analysis restored!")

✅ Complete analysis restored!


In [ ]:
with gr.Blocks(title="🌱 Crop Health Guardian") as final_app:

    gr.Markdown("""
    # 🌱 Crop Health Guardian
    ### Early Detection and Management of Crop Diseases and Pest Infestations
    """)

    language = gr.Dropdown(
        choices=language_options,
        value="English",
        label="🌐 Advisory Language"
    )

    image = gr.Image(
        type="pil",
        label="📷 Upload Crop Image"
    )

    with gr.Row():
        latitude = gr.Number(
            label="📍 Field Latitude",
            value=11.3410
        )

        longitude = gr.Number(
            label="📍 Field Longitude",
            value=77.7172
        )

    analyze_button = gr.Button(
        "🔍 Analyze Crop",
        variant="primary"
    )

    analysis_output = gr.Textbox(
        label="🤖 AI Crop Analysis",
        lines=16
    )

    advisory_output = gr.Textbox(
        label="💊 Multilingual Advisory",
        lines=10
    )

    validation_output = gr.Textbox(
        label="🔬 Expert Validation",
        lines=4
    )

    farmer_alert_output = gr.Textbox(
        label="🚨 Farmer Alert",
        lines=7
    )

    risk_output = gr.Textbox(
        label="📊 Overall Crop Risk",
        lines=3
    )

    map_output = gr.HTML(
        label="🗺️ Farmer Field Location"
    )

    status_output = gr.Textbox(
        label="⚙️ System Status",
        lines=2
    )

    analyze_button.click(
        complete_analysis,
        inputs=[
            image,
            language,
            latitude,
            longitude
        ],
        outputs=[
            analysis_output,
            advisory_output,
            validation_output,
            farmer_alert_output,
            risk_output,
            map_output,
            status_output
        ]
    )

print("✅ Dashboard created!")

✅ Dashboard created!


In [ ]:
final_app.launch(
    share=True,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d5c067bbaf5a3ea319.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import glob
from PIL import Image
import traceback

test_files = glob.glob(
    "/content/PlantVillage-Dataset/raw/color/*/*.JPG"
) + glob.glob(
    "/content/PlantVillage-Dataset/raw/color/*/*.jpg"
) + glob.glob(
    "/content/PlantVillage-Dataset/raw/color/*/*.png"
)

print("Test images found:", len(test_files))

test_img = Image.open(test_files[0]).convert("RGB")

try:
    result = complete_analysis(
        test_img,
        "English",
        11.3410,
        77.7172
    )
    print("✅ Backend test completed!")
    print(result[0][:1000])

except Exception as e:
    print("❌ BACKEND ERROR:")
    traceback.print_exc()

Test images found: 0


IndexError: list index out of range

In [ ]:
from PIL import Image
import numpy as np
import traceback

# Create a simple test image
test_img = Image.fromarray(
    np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
)

try:
    result = complete_analysis(
        test_img,
        "English",
        11.3410,
        77.7172
    )

    print("✅ BACKEND TEST PASSED!")
    print(result[0][:1000])

except Exception:
    print("❌ BACKEND ERROR:")
    traceback.print_exc()

❌ BACKEND ERROR:


Traceback (most recent call last):
  File "/tmp/ipykernel_1086/2903764336.py", line 11, in <cell line: 0>
    result = complete_analysis(
        test_img,
    ...<2 lines>...
        77.7172
    )
  File "/tmp/ipykernel_1086/36684947.py", line 87, in complete_analysis
    advisory = get_multilingual_advisory(
               ^^^^^^^^^^^^^^^^^^^^^^^^^
NameError: name 'get_multilingual_advisory' is not defined


In [ ]:
def get_multilingual_advisory(disease, pest, weather_risk, language):

    if language == "English":
        return generate_advisory(disease, pest, weather_risk)

    translations = {
        "தமிழ்": ("🌱 பயிர் சுகாதார ஆலோசனை",
                  "நோய்", "பூச்சி", "வானிலை அபாயம்",
                  "பாதிக்கப்பட்ட தாவரப் பகுதிகளை அகற்றி, பயிரை தொடர்ந்து கண்காணிக்கவும்."),

        "हिन्दी": ("🌱 फसल स्वास्थ्य सलाह",
                   "रोग", "कीट", "मौसम जोखिम",
                   "प्रभावित पौधों के हिस्सों को हटाएं और फसल की नियमित निगरानी करें।"),

        "తెలుగు": ("🌱 పంట ఆరోగ్య సలహా",
                    "వ్యాధి", "పురుగు", "వాతావరణ ప్రమాదం",
                    "ప్రభావిత మొక్కల భాగాలను తొలగించి పంటను పర్యవేక్షించండి."),

        "ಕನ್ನಡ": ("🌱 ಬೆಳೆ ಆರೋಗ್ಯ ಸಲಹೆ",
                   "ರೋಗ", "ಕೀಟ", "ಹವಾಮಾನ ಅಪಾಯ",
                   "ಬಾಧಿತ ಸಸ್ಯ ಭಾಗಗಳನ್ನು ತೆಗೆದುಹಾಕಿ ಮತ್ತು ಬೆಳೆಯನ್ನು ನಿಯಮಿತವಾಗಿ ಪರಿಶೀಲಿಸಿ."),

        "മലയാളം": ("🌱 വിള ആരോഗ്യ ഉപദേശം",
                    "രോഗം", "കീടം", "കാലാവസ്ഥാ അപകടസാധ്യത",
                    "ബാധിച്ച സസ്യഭാഗങ്ങൾ നീക്കം ചെയ്ത് വിള നിരീക്ഷിക്കുക."),

        "বাংলা": ("🌱 ফসল স্বাস্থ্য পরামর্শ",
                   "রোগ", "পোকা", "আবহাওয়ার ঝুঁকি",
                   "আক্রান্ত উদ্ভিদের অংশ সরিয়ে ফেলুন এবং নিয়মিত ফসল পর্যবেক্ষণ করুন।"),

        "मराठी": ("🌱 पीक आरोग्य सल्ला",
                   "रोग", "कीड", "हवामानाचा धोका",
                   "बाधित झाडांचे भाग काढून टाका आणि पिकाचे नियमित निरीक्षण करा."),

        "ગુજરાતી": ("🌱 પાક આરોગ્ય સલાહ",
                     "રોગ", "જીવાત", "હવામાન જોખમ",
                     "અસરગ્રસ્ત છોડના ભાગોને દૂર કરો અને પાકનું નિયમિત નિરીક્ષણ કરો."),

        "ਪੰਜਾਬੀ": ("🌱 ਫਸਲ ਸਿਹਤ ਸਲਾਹ",
                    "ਬਿਮਾਰੀ", "ਕੀੜਾ", "ਮੌਸਮ ਦਾ ਖਤਰਾ",
                    "ਪ੍ਰਭਾਵਿਤ ਪੌਦਿਆਂ ਦੇ ਹਿੱਸੇ ਹਟਾਓ ਅਤੇ ਫਸਲ ਦੀ ਨਿਯਮਿਤ ਨਿਗਰਾਨੀ ਕਰੋ."),

        "ଓଡ଼ିଆ": ("🌱 ଫସଲ ସ୍ୱାସ୍ଥ୍ୟ ପରାମର୍ଶ",
                   "ରୋଗ", "କୀଟ", "ପାଣିପାଗ ଜୋଖିମ",
                   "ଆକ୍ରାନ୍ତ ଗଛର ଅଂଶଗୁଡ଼ିକୁ ବାହାର କରନ୍ତୁ ଏବଂ ଫସଲକୁ ନିୟମିତ ନିରୀକ୍ଷଣ କରନ୍ତୁ.")
    }

    if language in translations:
        title, disease_label, pest_label, weather_label, advice = translations[language]

        return f"""
{title}

🦠 {disease_label}: {disease}
🐛 {pest_label}: {pest}
🌦️ {weather_label}: {weather_risk}

💊 பரிந்துரை / Advice:
{advice}

🐛 Pest management:
Use integrated pest management and continue regular monitoring.

🔬 Expert validation:
Contact an agricultural expert if the risk is high.
"""

    return generate_advisory(disease, pest, weather_risk)

print("✅ Multilingual advisory restored!")

✅ Multilingual advisory restored!


In [ ]:
try:
    result = complete_analysis(
        test_img,
        "English",
        11.3410,
        77.7172
    )

    print("✅ BACKEND TEST PASSED!")
    print(result[0][:1000])

except Exception:
    print("❌ BACKEND ERROR:")
    import traceback
    traceback.print_exc()

✅ BACKEND TEST PASSED!

🌱 CROP HEALTH ANALYSIS

🦠 Disease:
Corn_(maize)___healthy

📊 Disease Confidence:
62.86%

🐛 Pest:
No pest detected

📊 Pest Confidence:
0.00%

🌦️ LIVE WEATHER

🌡️ Temperature: 26.6 °C
💧 Humidity: 68 %
🌧️ Rainfall: 0.0 mm

⚠️ Weather Risk:
🟡 MEDIUM RISK

📊 OVERALL CROP RISK

Risk Score: 60/100
Risk Level: 🟡 MEDIUM CROP RISK

📍 FARMER LOCATION

Latitude: 11.341
Longitude: 77.7172



In [ ]:
final_app.launch(
    share=True,
    debug=False
)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d5c067bbaf5a3ea319.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import folium
import numpy as np
from sklearn.cluster import DBSCAN

# Sample crop-health reports
reports = [
    [11.3410, 77.7172, "Tomato Early Blight"],
    [11.3415, 77.7178, "Tomato Early Blight"],
    [11.3420, 77.7180, "Tomato Late Blight"],
    [11.3425, 77.7185, "Tomato Early Blight"],
    [11.3500, 77.7300, "Rice Leaf Hopper"],
    [11.3600, 77.7500, "Rice Gall Midge"]
]

coords = np.array([[r[0], r[1]] for r in reports])

# Detect nearby clusters
dbscan = DBSCAN(
    eps=0.00015,
    min_samples=3,
    metric="haversine"
)

labels = dbscan.fit_predict(np.radians(coords))

# Create map
hotspot_map = folium.Map(
    location=[11.345, 77.720],
    zoom_start=12
)

for i, report in enumerate(reports):

    lat, lon, disease = report

    if labels[i] >= 0:
        color = "red"
        popup = f"🔴 HOTSPOT<br>{disease}"
    else:
        color = "blue"
        popup = f"🔵 {disease}"

    folium.Marker(
        [lat, lon],
        popup=popup
    ).add_to(hotspot_map)

print("✅ Hotspot detection completed!")
print("Hotspot clusters:", len(set(labels)) - (1 if -1 in labels else 0))

hotspot_map

✅ Hotspot detection completed!
Hotspot clusters: 1


In [ ]:
import pandas as pd

report_df = pd.DataFrame(
    reports,
    columns=["Latitude", "Longitude", "Disease_or_Pest"]
)

report_df["Hotspot_ID"] = labels

report_df["Status"] = report_df["Hotspot_ID"].apply(
    lambda x: "Hotspot" if x >= 0 else "Individual Report"
)

report_df.to_csv(
    "crop_health_reports.csv",
    index=False
)

print("✅ Reports stored successfully!")
print("Total reports:", len(report_df))
print("\nSaved file: crop_health_reports.csv")

report_df

✅ Reports stored successfully!
Total reports: 6

Saved file: crop_health_reports.csv


,Latitude,Longitude,Disease_or_Pest,Hotspot_ID,Status
0,11.3410,77.7172,Tomato Early Blight,0,Hotspot
1,11.3415,77.7178,Tomato Early Blight,0,Hotspot
2,11.3420,77.7180,Tomato Late Blight,0,Hotspot
3,11.3425,77.7185,Tomato Early Blight,0,Hotspot
4,11.3500,77.7300,Rice Leaf Hopper,-1,Individual Report
5,11.3600,77.7500,Rice Gall Midge,-1,Individual Report


In [ ]:
import gradio as gr
import pandas as pd

def officer_dashboard():
    df = pd.read_csv("crop_health_reports.csv")

    total = len(df)
    hotspots = (df["Status"] == "Hotspot").sum()
    individual = (df["Status"] == "Individual Report").sum()

    distribution = (
        df["Disease_or_Pest"]
        .value_counts()
        .to_string()
    )

    return f"""
📊 AGRICULTURAL OFFICER DASHBOARD

📌 Total Reports: {total}

🔴 Hotspot Reports: {hotspots}

🔵 Individual Reports: {individual}

🦠 DISEASE / PEST REPORTS
--------------------------
{distribution}
"""

officer_app = gr.Interface(
    fn=officer_dashboard,
    inputs=[],
    outputs=gr.Textbox(
        label="📊 Officer Dashboard",
        lines=15
    ),
    title="🌾 Agricultural Officer Dashboard"
)

print("✅ Officer dashboard created!")

✅ Officer dashboard created!


In [ ]:
def expert_validation(disease, confidence):
    confidence = float(confidence)

    if confidence >= 0.70:
        return f"""
🔬 EXPERT VALIDATION

🦠 Prediction:
{disease}

📊 AI Confidence:
{confidence * 100:.2f}%

✅ Prediction confidence is acceptable.
Expert validation is optional.
"""
    else:
        return f"""
🔬 EXPERT VALIDATION

🦠 Prediction:
{disease}

📊 AI Confidence:
{confidence * 100:.2f}%

⚠️ Low-confidence prediction.
👨‍🌾 Agricultural expert validation is recommended.
"""

print("✅ Expert validation workflow created!")

✅ Expert validation workflow created!


In [ ]:
officer_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c5860f568d08ef32bf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import os
import re
import pandas as pd
import gradio as gr
import folium
import numpy as np
from sklearn.cluster import DBSCAN

REPORT_FILE = "sih_crop_health_reports.csv"


# =========================================================
# CONNECTED ANALYSIS
# =========================================================

def connected_analysis(img, language, latitude, longitude):

    # Run your existing AI analysis
    result = complete_analysis(
        img,
        language,
        latitude,
        longitude
    )

    analysis = result[0]
    advisory = result[1]
    validation = result[2]
    farmer_alert = result[3]
    risk_output = result[4]
    map_html = result[5]
    status = result[6]

    # Extract important values from AI result
    disease_match = re.search(
        r"🦠 Disease:\s*\n(.+)",
        analysis
    )

    disease_conf_match = re.search(
        r"📊 Disease Confidence:\s*\n([\d.]+)%",
        analysis
    )

    pest_match = re.search(
        r"🐛 Pest:\s*\n(.+)",
        analysis
    )

    pest_conf_match = re.search(
        r"📊 Pest Confidence:\s*\n([\d.]+)%",
        analysis
    )

    temperature_match = re.search(
        r"🌡️ Temperature:\s*([\d.]+)",
        analysis
    )

    humidity_match = re.search(
        r"💧 Humidity:\s*([\d.]+)",
        analysis
    )

    rainfall_match = re.search(
        r"🌧️ Rainfall:\s*([\d.]+)",
        analysis
    )

    weather_match = re.search(
        r"⚠️ Weather Risk:\s*\n(.+)",
        analysis
    )

    score_match = re.search(
        r"Risk Score:\s*(\d+)/100",
        analysis
    )

    risk_match = re.search(
        r"Risk Level:\s*(.+)",
        analysis
    )

    disease = (
        disease_match.group(1)
        if disease_match else "Unknown"
    )

    disease_conf = (
        float(disease_conf_match.group(1))
        if disease_conf_match else 0
    )

    pest = (
        pest_match.group(1)
        if pest_match else "No pest detected"
    )

    pest_conf = (
        float(pest_conf_match.group(1))
        if pest_conf_match else 0
    )

    temperature = (
        float(temperature_match.group(1))
        if temperature_match else None
    )

    humidity = (
        float(humidity_match.group(1))
        if humidity_match else None
    )

    rainfall = (
        float(rainfall_match.group(1))
        if rainfall_match else None
    )

    weather_risk = (
        weather_match.group(1)
        if weather_match else "Unknown"
    )

    risk_score = (
        int(score_match.group(1))
        if score_match else 0
    )

    risk_level = (
        risk_match.group(1)
        if risk_match else "Unknown"
    )

    # =====================================================
    # SAVE COMPLETE REPORT
    # =====================================================

    new_report = pd.DataFrame([{
        "Latitude": latitude,
        "Longitude": longitude,
        "Language": language,
        "Disease": disease,
        "Disease_Confidence": disease_conf,
        "Pest": pest,
        "Pest_Confidence": pest_conf,
        "Temperature_C": temperature,
        "Humidity_Percent": humidity,
        "Rainfall_mm": rainfall,
        "Weather_Risk": weather_risk,
        "Risk_Score": risk_score,
        "Risk_Level": risk_level,
        "Expert_Validation": validation,
        "Farmer_Alert": farmer_alert,
        "Advisory": advisory
    }])

    if os.path.exists(REPORT_FILE):
        old_reports = pd.read_csv(REPORT_FILE)
        all_reports = pd.concat(
            [old_reports, new_report],
            ignore_index=True
        )
    else:
        all_reports = new_report

    all_reports.to_csv(
        REPORT_FILE,
        index=False
    )

    return (
        analysis,
        advisory,
        validation,
        farmer_alert,
        risk_output,
        map_html,
        status
    )


# =========================================================
# FARMER DASHBOARD
# =========================================================

with gr.Blocks(title="🌱 SIH Crop Health Guardian") as final_app:

    gr.Markdown("""
    # 🌱 Crop Health Guardian
    ### SIH 2026 — Early Detection and Management of Crop Diseases and Pest Infestations
    """)

    language = gr.Dropdown(
        choices=language_options,
        value="English",
        label="🌐 Advisory Language"
    )

    image = gr.Image(
        type="pil",
        label="📷 Upload Crop Image"
    )

    with gr.Row():

        latitude = gr.Number(
            label="📍 Field Latitude",
            value=11.3410
        )

        longitude = gr.Number(
            label="📍 Field Longitude",
            value=77.7172
        )

    analyze_button = gr.Button(
        "🔍 Analyze Crop",
        variant="primary"
    )

    analysis_output = gr.Textbox(
        label="🤖 AI Crop Analysis",
        lines=16
    )

    advisory_output = gr.Textbox(
        label="💊 Multilingual Advisory",
        lines=10
    )

    validation_output = gr.Textbox(
        label="🔬 Expert Validation",
        lines=4
    )

    farmer_alert_output = gr.Textbox(
        label="🚨 Farmer Alert",
        lines=7
    )

    risk_output = gr.Textbox(
        label="📊 Overall Crop Risk",
        lines=3
    )

    map_output = gr.HTML()

    status_output = gr.Textbox(
        label="⚙️ System Status",
        lines=2
    )

    analyze_button.click(
        connected_analysis,
        inputs=[
            image,
            language,
            latitude,
            longitude
        ],
        outputs=[
            analysis_output,
            advisory_output,
            validation_output,
            farmer_alert_output,
            risk_output,
            map_output,
            status_output
        ]
    )


# =========================================================
# CONNECTED OFFICER DASHBOARD
# =========================================================

def officer_dashboard():

    if not os.path.exists(REPORT_FILE):
        return (
            "⚠️ No farmer reports available yet.",
            pd.DataFrame(),
            ""
        )

    df = pd.read_csv(REPORT_FILE)

    total = len(df)

    high = len(
        df[df["Risk_Level"].astype(str).str.contains("HIGH")]
    )

    medium = len(
        df[df["Risk_Level"].astype(str).str.contains("MEDIUM")]
    )

    low = len(
        df[df["Risk_Level"].astype(str).str.contains("LOW")]
    )

    # ---------------------------------------------
    # HOTSPOT DETECTION FROM REAL SAVED REPORTS
    # ---------------------------------------------

    if len(df) >= 3:

        coords = np.radians(
            df[["Latitude", "Longitude"]].values
        )

        cluster_model = DBSCAN(
            eps=0.00015,
            min_samples=3,
            metric="haversine"
        )

        labels = cluster_model.fit_predict(coords)

        df["Hotspot_ID"] = labels

        hotspot_count = len(
            set(labels) - {-1}
        )

    else:

        df["Hotspot_ID"] = -1
        hotspot_count = 0

    # ---------------------------------------------
    # HOTSPOT MAP
    # ---------------------------------------------

    center_lat = df["Latitude"].mean()
    center_lon = df["Longitude"].mean()

    officer_map = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=11
    )

    for _, row in df.iterrows():

        is_hotspot = row["Hotspot_ID"] >= 0

        color = "red" if is_hotspot else "blue"

        folium.Marker(
            [
                row["Latitude"],
                row["Longitude"]
            ],
            popup=f"""
            🦠 Disease: {row['Disease']}<br>
            🐛 Pest: {row['Pest']}<br>
            📊 Risk: {row['Risk_Level']}<br>
            🔴 Hotspot: {is_hotspot}
            """,
            icon=folium.Icon(color=color)
        ).add_to(officer_map)

    summary = f"""
# 📊 AGRICULTURAL OFFICER DASHBOARD

📌 Total Farmer Reports: {total}

🔴 High Risk Reports: {high}

🟡 Medium Risk Reports: {medium}

🟢 Low Risk Reports: {low}

🗺️ Detected Hotspots: {hotspot_count}

🦠 Most Reported Diseases/Pests:
{df['Disease'].value_counts().head(5).to_string()}
"""

    return (
        summary,
        df,
        officer_map._repr_html_()
    )


with gr.Blocks(
    title="🌾 SIH Agricultural Officer Dashboard"
) as officer_app:

    gr.Markdown("""
    # 🌾 Agricultural Officer Dashboard
    ### SIH 2026 — Crop Health Monitoring
    """)

    refresh_button = gr.Button(
        "🔄 Refresh Dashboard"
    )

    summary_output = gr.Markdown()

    reports_output = gr.Dataframe(
        label="📋 Farmer Reports",
        interactive=False
    )

    map_output = gr.HTML()

    refresh_button.click(
        officer_dashboard,
        inputs=[],
        outputs=[
            summary_output,
            reports_output,
            map_output
        ]
    )

print("✅ COMPLETE CONNECTED SIH SYSTEM CREATED!")
print("🌱 Farmer Dashboard: final_app")
print("🌾 Officer Dashboard: officer_app")
print("💾 Reports file:", REPORT_FILE)

✅ COMPLETE CONNECTED SIH SYSTEM CREATED!
🌱 Farmer Dashboard: final_app
🌾 Officer Dashboard: officer_app
💾 Reports file: sih_crop_health_reports.csv


In [ ]:
import os

print("📁 Reports file exists:", os.path.exists(REPORT_FILE))

if os.path.exists(REPORT_FILE):
    df = pd.read_csv(REPORT_FILE)
    print("📊 Saved reports:", len(df))
else:
    print("ℹ️ No reports yet — this is normal.")

📁 Reports file exists: False
ℹ️ No reports yet — this is normal.


In [ ]:
final_app.launch(
    share=True,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://97e419889d446ce571.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import os
import pandas as pd

print("📁 Reports file exists:", os.path.exists(REPORT_FILE))

if os.path.exists(REPORT_FILE):
    df = pd.read_csv(REPORT_FILE)
    print("📊 Total saved reports:", len(df))
    print(df.tail(1))

📁 Reports file exists: True
📊 Total saved reports: 1
   Latitude  Longitude Language                   Disease  Disease_Confidence  \
0    11.341    77.7172  English  Strawberry___Leaf_scorch               54.33   

               Pest  Pest_Confidence  Temperature_C  Humidity_Percent  \
0  Rice Leaf Hopper            71.34           26.5              69.0   

   Rainfall_mm   Weather_Risk  Risk_Score        Risk_Level  \
0          0.0  🟡 MEDIUM RISK          90  🔴 HIGH CROP RISK   

                                   Expert_Validation  \
0  ⚠️ Uncertain prediction\n🔬 Expert validation r...   

                                        Farmer_Alert  \
0  \n🚨 FARMER ALERT\n\n⚠️ High crop risk detected...   

                                            Advisory  
0  🦠 Disease: Strawberry___Leaf_scorch\nRemove af...  


In [ ]:
officer_app.launch(
    share=True,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a2261554e0c16c79b3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import requests

def get_weather_forecast(latitude, longitude):

    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={latitude}&longitude={longitude}"
        f"&daily=temperature_2m_max,temperature_2m_min,"
        f"precipitation_sum,rain_sum"
        f"&forecast_days=3"
        f"&timezone=auto"
    )

    response = requests.get(url, timeout=10)
    data = response.json()

    daily = data["daily"]

    forecast = []

    for i in range(len(daily["time"])):
        forecast.append({
            "Date": daily["time"][i],
            "Max Temperature": daily["temperature_2m_max"][i],
            "Min Temperature": daily["temperature_2m_min"][i],
            "Rainfall": daily["rain_sum"][i]
        })

    return forecast

print("✅ Weather forecasting function created!")

✅ Weather forecasting function created!


In [ ]:
import gradio as gr

with gr.Blocks() as location_test_app:

    gr.Markdown("""
    ## 📍 Farmer Location

    Please allow location access so the system can automatically
    detect your field location.
    """)

    location_output = gr.Textbox(
        label="📍 Detected Location",
        placeholder="Waiting for location permission..."
    )

    gr.HTML("""
    <button onclick="getFarmerLocation()"
            style="padding:10px 18px; font-size:16px;">
        📍 Allow My Location
    </button>

    <script>
    function getFarmerLocation() {

        if (!navigator.geolocation) {
            alert("Location is not supported by this browser.");
            return;
        }

        navigator.geolocation.getCurrentPosition(
            function(position) {

                let lat = position.coords.latitude;
                let lon = position.coords.longitude;

                alert(
                    "📍 Location detected successfully!\\n" +
                    "Latitude: " + lat.toFixed(6) +
                    "\\nLongitude: " + lon.toFixed(6)
                );
            },

            function(error) {

                if (error.code === 1) {
                    alert(
                        "⚠️ Location permission denied. " +
                        "Please turn ON location permission in your browser."
                    );
                } else {
                    alert("⚠️ Unable to detect your location.");
                }
            }
        );
    }
    </script>
    """)

print("✅ Farmer location permission module created!")

✅ Farmer location permission module created!


/usr/local/lib/python3.13/dist-packages/gradio/components/html.py:208: UserWarning: A `<script>` tag was found in the content of a `gr.HTML` component. Browsers do not execute `<script>` tags inserted via `innerHTML`, so this script will not run. Use the `head` parameter to load external libraries (e.g. `head='<script src="..."></script>'`); `head` content is injected and loaded before `js_on_load` runs. Then, if needed, put code that uses those libraries in the `js_on_load` parameter, which executes when the component renders. See https://gradio.app/guides/custom-HTML-components for details.
  _warn_if_script_tag(value)


In [ ]:
location_test_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cbe39ad5d3e1f43a29.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

def show_location(latitude, longitude):
    return f"""
📍 Location detected successfully!

Latitude: {latitude}
Longitude: {longitude}
"""

with gr.Blocks() as location_test_app:

    gr.Markdown("## 📍 Farmer Location")

    latitude = gr.Textbox(
        label="Latitude",
        visible=False
    )

    longitude = gr.Textbox(
        label="Longitude",
        visible=False
    )

    location_result = gr.Markdown(
        "Click the button and allow location access."
    )

    location_button = gr.Button(
        "📍 Allow My Location"
    )

    location_button.click(
        fn=show_location,
        inputs=[latitude, longitude],
        outputs=location_result,
        js="""
        async () => {
            return await new Promise((resolve) => {
                if (!navigator.geolocation) {
                    resolve(["Location not supported", ""]);
                    return;
                }

                navigator.geolocation.getCurrentPosition(
                    (position) => {
                        resolve([
                            position.coords.latitude.toString(),
                            position.coords.longitude.toString()
                        ]);
                    },
                    () => {
                        resolve(["Location permission denied", ""]);
                    }
                );
            });
        }
        """
    )

print("✅ Location test created!")

✅ Location test created!


In [ ]:
location_test_app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5437165f59cac39459.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
latitude = 11.3410
longitude = 77.7172

forecast = get_weather_forecast(latitude, longitude)

print("🌦️ 3-DAY WEATHER FORECAST")
print("=" * 35)

for day in forecast:
    print(
        f"\n📅 Date: {day['Date']}"
        f"\n🌡️ Max Temperature: {day['Max Temperature']} °C"
        f"\n🌡️ Min Temperature: {day['Min Temperature']} °C"
        f"\n🌧️ Rainfall: {day['Rainfall']} mm"
    )

🌦️ 3-DAY WEATHER FORECAST

📅 Date: 2026-09-24
🌡️ Max Temperature: 35.2 °C
🌡️ Min Temperature: 24.3 °C
🌧️ Rainfall: 0.0 mm

📅 Date: 2026-09-25
🌡️ Max Temperature: 33.9 °C
🌡️ Min Temperature: 25.1 °C
🌧️ Rainfall: 0.4 mm

📅 Date: 2026-09-26
🌡️ Max Temperature: 36.6 °C
🌡️ Min Temperature: 23.0 °C
🌧️ Rainfall: 0.2 mm


In [ ]:
def location_weather(latitude, longitude):

    try:
        latitude = float(latitude)
        longitude = float(longitude)

        forecast = get_weather_forecast(
            latitude,
            longitude
        )

        output = "🌦️ WEATHER FORECAST FOR FARMER LOCATION\n\n"

        for day in forecast:
            output += (
                f"📅 {day['Date']}\n"
                f"🌡️ Max: {day['Max Temperature']} °C\n"
                f"🌡️ Min: {day['Min Temperature']} °C\n"
                f"🌧️ Rainfall: {day['Rainfall']} mm\n\n"
            )

        return output

    except Exception as e:
        return f"⚠️ Weather forecast unavailable: {e}"

print("✅ Location → Weather connection created!")

✅ Location → Weather connection created!


In [ ]:
location_weather(
    11.3410,
    77.7172
)

'🌦️ WEATHER FORECAST FOR FARMER LOCATION\n\n📅 2026-09-24\n🌡️ Max: 35.2 °C\n🌡️ Min: 24.3 °C\n🌧️ Rainfall: 0.0 mm\n\n📅 2026-09-25\n🌡️ Max: 33.9 °C\n🌡️ Min: 25.1 °C\n🌧️ Rainfall: 0.4 mm\n\n📅 2026-09-26\n🌡️ Max: 36.6 °C\n🌡️ Min: 23.0 °C\n🌧️ Rainfall: 0.2 mm\n\n'

In [ ]:
import gradio as gr

with gr.Blocks(title="🌱 Crop Health Guardian") as location_demo:

    gr.Markdown("""
    # 🌱 Crop Health Guardian
    ### 📍 Automatic Farmer Location
    """)

    latitude = gr.Number(
        label="Latitude",
        value=None,
        interactive=False
    )

    longitude = gr.Number(
        label="Longitude",
        value=None,
        interactive=False
    )

    location_status = gr.Markdown(
        "📍 Click the button and allow location access."
    )

    location_button = gr.Button(
        "📍 Allow My Location",
        variant="primary"
    )

    def location_success(lat, lon):
        return (
            lat,
            lon,
            f"✅ Location detected successfully!\n\n"
            f"📍 Latitude: `{lat}`\n"
            f"📍 Longitude: `{lon}`"
        )

    location_button.click(
        fn=location_success,
        inputs=[latitude, longitude],
        outputs=[latitude, longitude, location_status],
        js="""
        async () => {
            return await new Promise((resolve) => {

                if (!navigator.geolocation) {
                    resolve([null, null]);
                    return;
                }

                navigator.geolocation.getCurrentPosition(
                    (position) => {

                        resolve([
                            position.coords.latitude,
                            position.coords.longitude
                        ]);

                    },

                    () => {
                        alert(
                            "⚠️ Location permission was denied. " +
                            "Please allow location access in your browser."
                        );

                        resolve([null, null]);
                    }
                );

            });
        }
        """
    )

print("✅ Automatic farmer location integration prepared!")

✅ Automatic farmer location integration prepared!


In [ ]:
print("Latitude:", latitude.value if hasattr(latitude, "value") else "ready")
print("Longitude:", longitude.value if hasattr(longitude, "value") else "ready")

Latitude: None
Longitude: None


In [ ]:
import gradio as gr

with gr.Blocks(title="📍 Location Test") as app:

    gr.Markdown("## 📍 Automatic Farmer Location")

    latitude = gr.Textbox(
        label="Latitude",
        placeholder="Waiting for location..."
    )

    longitude = gr.Textbox(
        label="Longitude",
        placeholder="Waiting for location..."
    )

    location_button = gr.Button(
        "📍 Allow My Location",
        variant="primary"
    )

    location_button.click(
        fn=None,
        inputs=[],
        outputs=[latitude, longitude],
        js="""
        async () => {

            return await new Promise((resolve) => {

                if (!navigator.geolocation) {
                    alert("❌ Geolocation is not supported.");
                    resolve(["", ""]);
                    return;
                }

                navigator.geolocation.getCurrentPosition(
                    (position) => {

                        const lat = position.coords.latitude;
                        const lon = position.coords.longitude;

                        resolve([
                            lat.toFixed(6),
                            lon.toFixed(6)
                        ]);
                    },

                    (error) => {

                        alert(
                            "⚠️ Location permission denied or unavailable."
                        );

                        resolve(["", ""]);
                    }
                );

            });
        }
        """
    )

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d3aca0553657b87a7c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
latitude = gr.Number(...)
longitude = gr.Number(...)

In [ ]:
import gradio as gr

with gr.Blocks() as location_test:

    gr.Markdown("## 📍 Automatic Farmer Location")

    latitude = gr.Textbox(label="Latitude")
    longitude = gr.Textbox(label="Longitude")

    location_button = gr.Button(
        "📍 Allow My Location",
        variant="primary"
    )

    location_button.click(
        fn=None,
        inputs=[],
        outputs=[latitude, longitude],
        js="""
        async () => {
            return await new Promise((resolve) => {

                navigator.geolocation.getCurrentPosition(
                    (position) => {
                        resolve([
                            position.coords.latitude.toFixed(6),
                            position.coords.longitude.toFixed(6)
                        ]);
                    },
                    () => {
                        alert("⚠️ Location permission denied.");
                        resolve(["", ""]);
                    }
                );

            });
        }
        """
    )

location_test.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cc700b1a0d58eb3a08.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
latitude = ...
longitude = ...

In [ ]:
latitude = gr.Number(...)
longitude = gr.Number(...)

In [ ]:
import gradio as gr

# ==============================
# CONNECTED ANALYSIS
# ==============================

def connected_analysis(img, language, latitude, longitude):

    # Convert automatic location values to numbers
    try:
        latitude = float(latitude)
        longitude = float(longitude)
    except:
        return (
            "⚠️ Please click 'Allow My Location' first.",
            "",
            "",
            "",
            "",
            None,
            "❌ Location not detected"
        )

    # Your existing analysis
    result = complete_analysis(
        img,
        language,
        latitude,
        longitude
    )

    return result


# ==============================
# FINAL FARMER DASHBOARD
# ==============================

with gr.Blocks(title="🌱 Crop Health Guardian") as final_app:

    gr.Markdown("""
    # 🌱 Crop Health Guardian
    ### AI-Powered Crop Disease & Pest Management
    """)

    language = gr.Dropdown(
        choices=language_options,
        value="English",
        label="🌐 Select Language"
    )

    crop_image = gr.Image(
        type="pil",
        label="📷 Upload Crop Image"
    )

    # ==============================
    # AUTOMATIC LOCATION
    # ==============================

    gr.Markdown("### 📍 Farmer Location")

    latitude = gr.Textbox(
        label="Latitude",
        value="",
        interactive=False
    )

    longitude = gr.Textbox(
        label="Longitude",
        value="",
        interactive=False
    )

    location_button = gr.Button(
        "📍 Allow My Location",
        variant="primary"
    )

    location_button.click(
        fn=None,
        inputs=[],
        outputs=[latitude, longitude],
        js="""
        async () => {
            return await new Promise((resolve) => {

                if (!navigator.geolocation) {
                    alert("❌ Location is not supported.");
                    resolve(["", ""]);
                    return;
                }

                navigator.geolocation.getCurrentPosition(
                    (position) => {

                        resolve([
                            position.coords.latitude.toFixed(6),
                            position.coords.longitude.toFixed(6)
                        ]);

                    },

                    () => {

                        alert(
                            "⚠️ Location permission denied. " +
                            "Please allow location access."
                        );

                        resolve(["", ""]);
                    }
                );

            });
        }
        """
    )

    # ==============================
    # ANALYZE
    # ==============================

    analyze_button = gr.Button(
        "🔍 Analyze Crop",
        variant="primary"
    )

    analysis_output = gr.Textbox(
        label="🌱 AI Analysis",
        lines=8
    )

    advisory_output = gr.Textbox(
        label="👨‍🌾 Farmer Advisory",
        lines=8
    )

    validation_output = gr.Textbox(
        label="👨‍🔬 Expert Validation",
        lines=5
    )

    alert_output = gr.Textbox(
        label="🚨 Farmer Alert",
        lines=5
    )

    risk_output = gr.Textbox(
        label="⚠️ Overall Crop Risk",
        lines=3
    )

    map_output = gr.HTML(
        label="🗺️ Crop Risk Map"
    )

    status_output = gr.Textbox(
        label="📋 Status"
    )

    analyze_button.click(
        fn=connected_analysis,
        inputs=[
            crop_image,
            language,
            latitude,
            longitude
        ],
        outputs=[
            analysis_output,
            advisory_output,
            validation_output,
            alert_output,
            risk_output,
            map_output,
            status_output
        ]
    )


# ==============================
# LAUNCH
# ==============================

final_app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a123d44afdff6b45f1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def get_location_weather_forecast(latitude, longitude):

    try:
        latitude = float(latitude)
        longitude = float(longitude)

        forecast = get_weather_forecast(
            latitude,
            longitude
        )

        output = "🌦️ 3-DAY WEATHER FORECAST\n\n"

        for day in forecast:
            output += (
                f"📅 {day['Date']}\n"
                f"🌡️ Max Temperature: {day['Max Temperature']} °C\n"
                f"🌡️ Min Temperature: {day['Min Temperature']} °C\n"
                f"🌧️ Expected Rainfall: {day['Rainfall']} mm\n\n"
            )

        return output

    except Exception as e:
        return f"⚠️ Weather forecast unavailable: {e}"


# Quick test using your automatically detected location
print(
    get_location_weather_forecast(
        11.3410,
        77.7172
    )
)

🌦️ 3-DAY WEATHER FORECAST

📅 2026-09-24
🌡️ Max Temperature: 35.2 °C
🌡️ Min Temperature: 24.3 °C
🌧️ Expected Rainfall: 0.0 mm

📅 2026-09-25
🌡️ Max Temperature: 33.9 °C
🌡️ Min Temperature: 25.1 °C
🌧️ Expected Rainfall: 0.4 mm

📅 2026-09-26
🌡️ Max Temperature: 36.6 °C
🌡️ Min Temperature: 23.0 °C
🌧️ Expected Rainfall: 0.2 mm




In [ ]:
import gradio as gr

# ==============================
# WEATHER FORECAST
# ==============================

def show_weather_forecast(latitude, longitude):
    try:
        return get_location_weather_forecast(latitude, longitude)
    except Exception as e:
        return f"⚠️ Weather forecast unavailable: {e}"


# ==============================
# FINAL FARMER DASHBOARD
# ==============================

with gr.Blocks(title="🌱 Crop Health Guardian") as final_app:

    gr.Markdown("""
    # 🌱 Crop Health Guardian
    ### AI-Powered Crop Disease & Pest Management
    """)

    language = gr.Dropdown(
        choices=language_options,
        value="English",
        label="🌐 Select Language"
    )

    crop_image = gr.Image(
        type="pil",
        label="📷 Upload Crop Image"
    )

    # 📍 AUTOMATIC LOCATION

    gr.Markdown("### 📍 Farmer Location")

    latitude = gr.Textbox(
        label="Latitude",
        value="",
        interactive=False
    )

    longitude = gr.Textbox(
        label="Longitude",
        value="",
        interactive=False
    )

    location_button = gr.Button(
        "📍 Allow My Location",
        variant="primary"
    )

    # 🌦️ WEATHER FORECAST

    weather_forecast_output = gr.Textbox(
        label="🌦️ 3-Day Weather Forecast",
        lines=10,
        interactive=False
    )

    # 🔍 ANALYZE

    analyze_button = gr.Button(
        "🔍 Analyze Crop",
        variant="primary"
    )

    analysis_output = gr.Textbox(
        label="🌱 AI Analysis",
        lines=8
    )

    advisory_output = gr.Textbox(
        label="👨‍🌾 Farmer Advisory",
        lines=8
    )

    validation_output = gr.Textbox(
        label="👨‍🔬 Expert Validation",
        lines=5
    )

    alert_output = gr.Textbox(
        label="🚨 Farmer Alert",
        lines=5
    )

    risk_output = gr.Textbox(
        label="⚠️ Overall Crop Risk",
        lines=3
    )

    map_output = gr.HTML(
        label="🗺️ Crop Risk Map"
    )

    status_output = gr.Textbox(
        label="📋 Status"
    )

    # ==============================
    # LOCATION BUTTON
    # ==============================

    location_button.click(
        fn=None,
        inputs=[],
        outputs=[latitude, longitude],
        js="""
        async () => {

            return await new Promise((resolve) => {

                if (!navigator.geolocation) {
                    alert("❌ Location is not supported.");
                    resolve(["", ""]);
                    return;
                }

                navigator.geolocation.getCurrentPosition(
                    (position) => {

                        resolve([
                            position.coords.latitude.toFixed(6),
                            position.coords.longitude.toFixed(6)
                        ]);

                    },

                    () => {

                        alert(
                            "⚠️ Location permission denied. " +
                            "Please allow location access."
                        );

                        resolve(["", ""]);
                    }
                );

            });

        }
        """
    )

    # ==============================
    # WEATHER BUTTON CONNECTION
    # ==============================

    location_button.click(
        fn=show_weather_forecast,
        inputs=[latitude, longitude],
        outputs=weather_forecast_output
    )

    # ==============================
    # ANALYZE CONNECTION
    # ==============================

    analyze_button.click(
        fn=connected_analysis,
        inputs=[
            crop_image,
            language,
            latitude,
            longitude
        ],
        outputs=[
            analysis_output,
            advisory_output,
            validation_output,
            alert_output,
            risk_output,
            map_output,
            status_output
        ]
    )


final_app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://04c6963f07c80abce1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# ============================================
# 🌱 IPM + SAFE INPUT RECOMMENDATION MODULE
# ============================================

IPM_DATABASE = {

    "Early blight": {
        "ipm": [
            "Remove infected leaves and destroy them safely.",
            "Avoid overhead irrigation.",
            "Maintain proper spacing for good air circulation.",
            "Rotate crops where possible."
        ],
        "safe_input": [
            "Use approved fungicides only according to the product label.",
            "Prefer bio-control options such as Trichoderma where suitable.",
            "Follow the recommended dose and waiting period."
        ]
    },

    "Late blight": {
        "ipm": [
            "Remove severely infected plant parts.",
            "Avoid prolonged leaf wetness.",
            "Improve field ventilation.",
            "Monitor the crop closely during wet weather."
        ],
        "safe_input": [
            "Use a locally approved fungicide when required.",
            "Follow label dosage and pre-harvest interval.",
            "Avoid unnecessary repeated chemical applications."
        ]
    },

    "Powdery mildew": {
        "ipm": [
            "Remove heavily infected leaves.",
            "Improve air circulation.",
            "Avoid excessive nitrogen application.",
            "Monitor new growth regularly."
        ],
        "safe_input": [
            "Use approved sulfur-based or other registered fungicides when appropriate.",
            "Follow the product label and crop-specific instructions."
        ]
    },

    "Rice blast": {
        "ipm": [
            "Use healthy and disease-free seed.",
            "Avoid excessive nitrogen fertilizer.",
            "Maintain balanced irrigation.",
            "Remove severely affected plant material."
        ],
        "safe_input": [
            "Use locally approved rice-blast management products when necessary.",
            "Follow label dosage and pre-harvest interval."
        ]
    }
}


def get_ipm_recommendation(disease_name):

    disease_name = str(disease_name)

    # Find matching disease
    matched_key = None

    for key in IPM_DATABASE:

        if key.lower() in disease_name.lower():
            matched_key = key
            break

    if matched_key is None:
        return (
            "🌱 IPM RECOMMENDATION\n\n"
            "• Monitor the crop regularly.\n"
            "• Remove severely infected plant parts if appropriate.\n"
            "• Maintain field hygiene and proper spacing.\n"
            "• Avoid unnecessary pesticide application.\n\n"
            "🛡️ SAFE INPUT\n\n"
            "Use only locally registered agricultural products "
            "according to the label and expert recommendation."
        )

    data = IPM_DATABASE[matched_key]

    output = f"🌱 IPM RECOMMENDATION — {matched_key}\n\n"

    for action in data["ipm"]:
        output += f"• {action}\n"

    output += "\n🛡️ SAFE INPUT RECOMMENDATION\n\n"

    for action in data["safe_input"]:
        output += f"• {action}\n"

    return output


print(get_ipm_recommendation("Early blight"))

🌱 IPM RECOMMENDATION — Early blight

• Remove infected leaves and destroy them safely.
• Avoid overhead irrigation.
• Maintain proper spacing for good air circulation.
• Rotate crops where possible.

🛡️ SAFE INPUT RECOMMENDATION

• Use approved fungicides only according to the product label.
• Prefer bio-control options such as Trichoderma where suitable.
• Follow the recommended dose and waiting period.



In [ ]:
# ============================================
# 🌱 CONNECT IPM TO FARMER DASHBOARD
# ============================================

def extract_disease_for_ipm(analysis_text):
    """
    Extract the detected disease name from the AI analysis.
    """

    text = str(analysis_text)

    for disease in IPM_DATABASE:

        if disease.lower() in text.lower():
            return get_ipm_recommendation(disease)

    return get_ipm_recommendation("Unknown")


# Test with an example
test_analysis = "Disease detected: Early blight"

print(extract_disease_for_ipm(test_analysis))

🌱 IPM RECOMMENDATION — Early blight

• Remove infected leaves and destroy them safely.
• Avoid overhead irrigation.
• Maintain proper spacing for good air circulation.
• Rotate crops where possible.

🛡️ SAFE INPUT RECOMMENDATION

• Use approved fungicides only according to the product label.
• Prefer bio-control options such as Trichoderma where suitable.
• Follow the recommended dose and waiting period.



In [ ]:
# ============================================
# 🧑‍🔬 EXPERT / LAB REFERRAL MODULE
# ============================================

def generate_referral(disease_confidence, overall_risk):

    try:
        confidence = float(disease_confidence)
    except:
        confidence = 0.0

    if confidence < 0.70:
        return """
🧑‍🔬 EXPERT / LAB REFERRAL

⚠️ AI diagnosis confidence is low.

Recommended action:
• Take a clear photo of the affected plant.
• Consult an agricultural extension officer.
• Submit a plant sample to an agricultural laboratory if required.
• Do not apply pesticides based only on the AI prediction.
"""

    if "HIGH" in str(overall_risk).upper():
        return """
🧑‍🔬 EXPERT / LAB REFERRAL

🔴 High crop risk detected.

Recommended action:
• Contact an agricultural expert for confirmation.
• Consider laboratory testing if the disease is unclear.
• Follow expert advice before applying chemical inputs.
• Continue monitoring the affected field.
"""

    return """
🧑‍🔬 EXPERT / LAB REFERRAL

✅ No immediate laboratory referral required.

Continue:
• Regular crop monitoring
• IPM practices
• Weather-risk monitoring
• Follow-up observation
"""


# ============================================
# CONNECT ANALYSIS → REFERRAL
# ============================================

def referral_from_analysis(analysis_text, risk_text):

    import re

    text = str(analysis_text)

    confidence = 0.0

    match = re.search(
        r'(\d+(?:\.\d+)?)\s*%',
        text
    )

    if match:
        confidence = float(match.group(1)) / 100

    return generate_referral(
        confidence,
        risk_text
    )


# ============================================
# TEST
# ============================================

print(
    referral_from_analysis(
        "Disease detected: Early blight (62%)",
        "🔴 HIGH CROP RISK"
    )
)


🧑‍🔬 EXPERT / LAB REFERRAL

⚠️ AI diagnosis confidence is low.

Recommended action:
• Take a clear photo of the affected plant.
• Consult an agricultural extension officer.
• Submit a plant sample to an agricultural laboratory if required.
• Do not apply pesticides based only on the AI prediction.



In [ ]:
# ============================================
# 📋 FOLLOW-UP MONITORING MODULE
# ============================================

import pandas as pd
from datetime import datetime

FOLLOWUP_FILE = "sih_crop_followup.csv"


def save_followup(
    disease,
    status,
    farmer_note=""
):

    record = {
        "Date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Disease": disease,
        "Follow-up Status": status,
        "Farmer Note": farmer_note
    }

    df = pd.DataFrame([record])

    try:
        old_df = pd.read_csv(FOLLOWUP_FILE)
        df = pd.concat([old_df, df], ignore_index=True)
    except FileNotFoundError:
        pass

    df.to_csv(FOLLOWUP_FILE, index=False)

    return "✅ Follow-up observation saved successfully."


# Test
print(
    save_followup(
        "Early blight",
        "Improving",
        "Symptoms reduced after treatment."
    )
)

✅ Follow-up observation saved successfully.


In [ ]:
# ============================================
# 📋 FOLLOW-UP MONITORING UI
# ============================================

import gradio as gr

with gr.Blocks(title="📋 Crop Follow-up") as followup_app:

    gr.Markdown("""
    ## 📋 Crop Follow-up Monitoring
    Record the condition of the crop after the initial diagnosis.
    """)

    disease_input = gr.Textbox(
        label="🌱 Detected Disease",
        placeholder="Example: Early blight"
    )

    followup_status = gr.Dropdown(
        choices=[
            "Improving",
            "No Change",
            "Worsening",
            "Recovered"
        ],
        label="📊 Crop Condition"
    )

    farmer_note = gr.Textbox(
        label="📝 Farmer Observation",
        placeholder="Describe what happened after treatment..."
    )

    save_button = gr.Button(
        "💾 Save Follow-up",
        variant="primary"
    )

    followup_result = gr.Textbox(
        label="Status",
        interactive=False
    )

    save_button.click(
        fn=save_followup,
        inputs=[
            disease_input,
            followup_status,
            farmer_note
        ],
        outputs=followup_result
    )

followup_app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://92a94f93558f712981.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import os
import pandas as pd

print("REPORT FILE EXISTS:", os.path.exists("sih_crop_health_reports.csv"))

if os.path.exists("sih_crop_health_reports.csv"):
    df = pd.read_csv("sih_crop_health_reports.csv")
    print("NUMBER OF REPORTS:", len(df))
    print("\nCOLUMNS:")
    print(df.columns.tolist())
    print("\nFIRST REPORT:")
    display(df.head(1))
else:
    print("⚠️ Report file not found.")

REPORT FILE EXISTS: True
NUMBER OF REPORTS: 1

COLUMNS:
['Latitude', 'Longitude', 'Language', 'Disease', 'Disease_Confidence', 'Pest', 'Pest_Confidence', 'Temperature_C', 'Humidity_Percent', 'Rainfall_mm', 'Weather_Risk', 'Risk_Score', 'Risk_Level', 'Expert_Validation', 'Farmer_Alert', 'Advisory']

FIRST REPORT:


,Latitude,Longitude,Language,Disease,Disease_Confidence,Pest,Pest_Confidence,Temperature_C,Humidity_Percent,Rainfall_mm,Weather_Risk,Risk_Score,Risk_Level,Expert_Validation,Farmer_Alert,Advisory
0,11.341,77.7172,English,Strawberry___Leaf_scorch,54.33,Rice Leaf Hopper,71.34,26.5,69.0,0.0,🟡 MEDIUM RISK,90,🔴 HIGH CROP RISK,⚠️ Uncertain prediction\n🔬 Expert validation r...,\n🚨 FARMER ALERT\n\n⚠️ High crop risk detected...,🦠 Disease: Strawberry___Leaf_scorch\nRemove af...


In [ ]:
import gradio as gr
import pandas as pd
from datetime import datetime
import os

REPORT_FILE = "sih_crop_health_reports.csv"
FOLLOWUP_FILE = "sih_crop_followup.csv"


def get_reports():

    if not os.path.exists(REPORT_FILE):
        return pd.DataFrame()

    return pd.read_csv(REPORT_FILE)


def load_previous_report(report_number):

    df = get_reports()

    if df.empty:
        return (
            "No report available",
            "",
            "",
            "",
            "",
            "",
            ""
        )

    try:
        index = int(report_number) - 1
        row = df.iloc[index]

        return (
            str(row["Disease"]),
            str(row["Disease_Confidence"]),
            str(row["Risk_Level"]),
            str(row["Risk_Score"]),
            str(row["Latitude"]),
            str(row["Longitude"]),
            str(row["Language"])
        )

    except Exception as e:

        return (
            f"Error: {e}",
            "",
            "",
            "",
            "",
            "",
            ""
        )


def save_followup(
    report_number,
    condition,
    observation
):

    df = get_reports()

    if df.empty:
        return "⚠️ No previous report found."

    try:

        index = int(report_number) - 1
        row = df.iloc[index]

        record = {
            "Followup_Date":
                datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

            "Report_Number":
                report_number,

            "Disease":
                row["Disease"],

            "Disease_Confidence":
                row["Disease_Confidence"],

            "Original_Risk_Level":
                row["Risk_Level"],

            "Original_Risk_Score":
                row["Risk_Score"],

            "Latitude":
                row["Latitude"],

            "Longitude":
                row["Longitude"],

            "Language":
                row["Language"],

            "Current_Crop_Condition":
                condition,

            "Farmer_Observation":
                observation
        }

        new_df = pd.DataFrame([record])

        if os.path.exists(FOLLOWUP_FILE):

            old_df = pd.read_csv(FOLLOWUP_FILE)

            new_df = pd.concat(
                [old_df, new_df],
                ignore_index=True
            )

        new_df.to_csv(
            FOLLOWUP_FILE,
            index=False
        )

        return "✅ Follow-up observation saved successfully."

    except Exception as e:

        return f"⚠️ Error: {e}"


# ---------------- UI ----------------

reports = get_reports()

report_numbers = [
    str(i + 1)
    for i in range(len(reports))
]


with gr.Blocks(
    title="Crop Follow-up Monitoring"
) as followup_app:

    gr.Markdown("""
    # 📋 Crop Follow-up Monitoring

    Select the previous farmer report.
    The system will automatically load the original diagnosis.
    """)

    report_number = gr.Dropdown(
        choices=report_numbers,
        label="Select Previous Report"
    )

    load_button = gr.Button(
        "🔄 Load Previous Report"
    )

    gr.Markdown(
        "### 🤖 Automatically Retrieved Information"
    )

    disease = gr.Textbox(
        label="Detected Disease",
        interactive=False
    )

    disease_confidence = gr.Textbox(
        label="Disease Confidence",
        interactive=False
    )

    risk_level = gr.Textbox(
        label="Original Risk Level",
        interactive=False
    )

    risk_score = gr.Textbox(
        label="Original Risk Score",
        interactive=False
    )

    latitude = gr.Textbox(
        label="Latitude",
        interactive=False
    )

    longitude = gr.Textbox(
        label="Longitude",
        interactive=False
    )

    language = gr.Textbox(
        label="Language",
        interactive=False
    )

    gr.Markdown(
        "### 👨‍🌾 Farmer Provides New Information"
    )

    condition = gr.Dropdown(
        choices=[
            "Improving",
            "No Change",
            "Worsening",
            "Recovered"
        ],
        label="Current Crop Condition"
    )

    observation = gr.Textbox(
        label="Farmer Observation",
        placeholder="Describe the current condition of the crop..."
    )

    save_button = gr.Button(
        "💾 Save Follow-up",
        variant="primary"
    )

    result = gr.Textbox(
        label="Status"
    )


    load_button.click(
        fn=load_previous_report,
        inputs=report_number,
        outputs=[
            disease,
            disease_confidence,
            risk_level,
            risk_score,
            latitude,
            longitude,
            language
        ]
    )


    save_button.click(
        fn=save_followup,
        inputs=[
            report_number,
            condition,
            observation
        ],
        outputs=result
    )


followup_app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b33e5816977da5308f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def generate_followup_feedback(condition):

    if condition == "Improving":
        return """
🟢 FOLLOW-UP STATUS

Crop condition is improving.

Recommended:
• Continue the advised IPM practices.
• Continue weather-risk monitoring.
• Observe the crop regularly.
"""

    elif condition == "No Change":
        return """
🟡 FOLLOW-UP STATUS

No significant change has been reported.

Recommended:
• Continue monitoring the crop.
• Follow the existing advisory.
• Reassess the crop after further observation.
"""

    elif condition == "Worsening":
        return """
🔴 FOLLOW-UP STATUS

Crop condition is worsening.

Recommended:
• Contact an agricultural expert.
• Consider laboratory confirmation if required.
• Avoid applying additional pesticides without proper guidance.
• Upload a new crop image for further AI analysis if available.
"""

    elif condition == "Recovered":
        return """
✅ FOLLOW-UP STATUS

Crop recovery has been reported.

Recommended:
• Continue regular crop monitoring.
• Maintain field hygiene.
• Record the recovery for future AI improvement.
"""

    return "Select the current crop condition."


print(generate_followup_feedback("Worsening"))


🔴 FOLLOW-UP STATUS

Crop condition is worsening.

Recommended:
• Contact an agricultural expert.
• Consider laboratory confirmation if required.
• Avoid applying additional pesticides without proper guidance.
• Upload a new crop image for further AI analysis if available.



In [ ]:
def save_followup_with_feedback(
    report_number,
    condition,
    observation
):

    df = get_reports()

    if df.empty:
        return "⚠️ No previous report found."

    try:
        index = int(report_number) - 1
        row = df.iloc[index]

        feedback = generate_followup_feedback(condition)

        record = {
            "Followup_Date":
                datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

            "Report_Number":
                report_number,

            "Disease":
                row["Disease"],

            "Disease_Confidence":
                row["Disease_Confidence"],

            "Original_Risk_Level":
                row["Risk_Level"],

            "Original_Risk_Score":
                row["Risk_Score"],

            "Latitude":
                row["Latitude"],

            "Longitude":
                row["Longitude"],

            "Language":
                row["Language"],

            "Current_Crop_Condition":
                condition,

            "Farmer_Observation":
                observation,

            "Followup_Feedback":
                feedback
        }

        new_df = pd.DataFrame([record])

        if os.path.exists(FOLLOWUP_FILE):
            old_df = pd.read_csv(FOLLOWUP_FILE)
            new_df = pd.concat(
                [old_df, new_df],
                ignore_index=True
            )

        new_df.to_csv(
            FOLLOWUP_FILE,
            index=False
        )

        return feedback

    except Exception as e:
        return f"⚠️ Error: {e}"

In [ ]:
def load_followup_history():

    if not os.path.exists(FOLLOWUP_FILE):
        return pd.DataFrame(
            columns=[
                "Followup_Date",
                "Report_Number",
                "Disease",
                "Disease_Confidence",
                "Original_Risk_Level",
                "Original_Risk_Score",
                "Latitude",
                "Longitude",
                "Language",
                "Current_Crop_Condition",
                "Farmer_Observation",
                "Followup_Feedback"
            ]
        )

    return pd.read_csv(FOLLOWUP_FILE)


with gr.Blocks(
    title="Follow-up History"
) as followup_history_app:

    gr.Markdown("""
    # 📋 Follow-up Monitoring History

    View the crop condition reported during follow-up.
    """)

    load_history_button = gr.Button(
        "🔄 Load Follow-up History"
    )

    history_table = gr.Dataframe(
        headers=None,
        interactive=False
    )

    load_history_button.click(
        fn=load_followup_history,
        inputs=[],
        outputs=history_table
    )


followup_history_app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ed0be1c34935a23882.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def record_followup_confirmation(
    report_number,
    field_confirmed,
    actual_disease="",
    farmer_note=""
):

    df = get_reports()

    if df.empty:
        return "⚠️ No previous report found."

    try:
        index = int(report_number) - 1
        row = df.iloc[index]

        predicted_disease = str(row["Disease"])

        save_ai_feedback(
            predicted_disease=predicted_disease,
            field_confirmed=field_confirmed,
            actual_disease=actual_disease,
            farmer_note=farmer_note
        )

        return "✅ Field confirmation saved for AI improvement."

    except Exception as e:
        return f"⚠️ Error: {e}"

In [ ]:
final_app.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://04c6963f07c80abce1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import pandas as pd

df = pd.read_csv("sih_crop_health_reports.csv")

print("✅ Total reports:", len(df))
print("\nLatest report:")
display(df.tail(1))

✅ Total reports: 1

Latest report:


,Latitude,Longitude,Language,Disease,Disease_Confidence,Pest,Pest_Confidence,Temperature_C,Humidity_Percent,Rainfall_mm,Weather_Risk,Risk_Score,Risk_Level,Expert_Validation,Farmer_Alert,Advisory
0,11.341,77.7172,English,Strawberry___Leaf_scorch,54.33,Rice Leaf Hopper,71.34,26.5,69.0,0.0,🟡 MEDIUM RISK,90,🔴 HIGH CROP RISK,⚠️ Uncertain prediction\n🔬 Expert validation r...,\n🚨 FARMER ALERT\n\n⚠️ High crop risk detected...,🦠 Disease: Strawberry___Leaf_scorch\nRemove af...


In [ ]:
import os
import pandas as pd

print("Report file:", os.path.abspath("sih_crop_health_reports.csv"))
print("Last modified:", os.path.getmtime("sih_crop_health_reports.csv"))

df = pd.read_csv("sih_crop_health_reports.csv")

print("Number of rows:", len(df))
print("Latest saved report:")
display(df.tail(1))

Report file: /content/sih_crop_health_reports.csv
Last modified: 1790265332.3152623
Number of rows: 1
Latest saved report:


,Latitude,Longitude,Language,Disease,Disease_Confidence,Pest,Pest_Confidence,Temperature_C,Humidity_Percent,Rainfall_mm,Weather_Risk,Risk_Score,Risk_Level,Expert_Validation,Farmer_Alert,Advisory
0,11.341,77.7172,English,Strawberry___Leaf_scorch,54.33,Rice Leaf Hopper,71.34,26.5,69.0,0.0,🟡 MEDIUM RISK,90,🔴 HIGH CROP RISK,⚠️ Uncertain prediction\n🔬 Expert validation r...,\n🚨 FARMER ALERT\n\n⚠️ High crop risk detected...,🦠 Disease: Strawberry___Leaf_scorch\nRemove af...


In [ ]:
officer_app.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a2261554e0c16c79b3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
followup_app.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b33e5816977da5308f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
followup_history_app.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ed0be1c34935a23882.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
print(
    record_followup_confirmation(
        report_number="1",
        field_confirmed="Yes",
        actual_disease="Strawberry___Leaf_scorch",
        farmer_note="Disease symptoms confirmed during follow-up."
    )
)

⚠️ Error: name 'save_ai_feedback' is not defined


In [ ]:
import pandas as pd
from datetime import datetime

FEEDBACK_FILE = "sih_ai_feedback.csv"

def save_ai_feedback(
    predicted_disease,
    field_confirmed,
    actual_disease="",
    farmer_note=""
):

    record = {
        "Date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Predicted Disease": predicted_disease,
        "Field Confirmed": field_confirmed,
        "Actual Disease": actual_disease,
        "Farmer Note": farmer_note
    }

    new_df = pd.DataFrame([record])

    try:
        old_df = pd.read_csv(FEEDBACK_FILE)
        new_df = pd.concat(
            [old_df, new_df],
            ignore_index=True
        )
    except FileNotFoundError:
        pass

    new_df.to_csv(
        FEEDBACK_FILE,
        index=False
    )

    return "✅ Field confirmation saved for AI improvement."

print("✅ save_ai_feedback function restored.")

✅ save_ai_feedback function restored.


In [ ]:
print(
    record_followup_confirmation(
        report_number="1",
        field_confirmed="Yes",
        actual_disease="Strawberry___Leaf_scorch",
        farmer_note="Disease symptoms confirmed during follow-up."
    )
)

✅ Field confirmation saved for AI improvement.


In [ ]:
import pandas as pd

feedback = pd.read_csv("sih_ai_feedback.csv")

print("✅ Total field confirmations:", len(feedback))
display(feedback.tail(1))

✅ Total field confirmations: 1


,Date,Predicted Disease,Field Confirmed,Actual Disease,Farmer Note
0,2026-09-24 17:07:42,Strawberry___Leaf_scorch,Yes,Strawberry___Leaf_scorch,Disease symptoms confirmed during follow-up.


In [ ]:
from google.colab import files
import os

backup_files = [
    "plant_disease_efficientnet_b0.pth",
    "best (1).pt",
    "sih_crop_health_reports.csv",
    "sih_crop_followup.csv",
    "sih_ai_feedback.csv"
]

for file in backup_files:
    if os.path.exists(file):
        print("✅", file)
        files.download(file)
    else:
        print("⚠️ Not found:", file)

✅ plant_disease_efficientnet_b0.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ best (1).pt


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ sih_ai_feedback.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# SIH 2026 – Crop Disease & Pest Management System
# SECTION 1: IMPORTS & SETUP
# ============================================================

import os
import re
import requests
import numpy as np
import pandas as pd
import torch
import folium
import gradio as gr

from datetime import datetime
from torchvision import transforms
from sklearn.cluster import DBSCAN
from PIL import Image

print("✅ Imports and setup completed")
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

✅ Imports and setup completed
PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4
